### Data Ingestion to Vector DB pipeline


In [20]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from pathlib import Path

In [21]:
from pathlib import Path

## Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f" Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

for i, doc in enumerate(all_pdf_documents):
    print(f"Page {i+1}: {len(doc.page_content)} characters")


print(repr(all_pdf_documents[0].page_content))    

Found 1 PDF files to process

Processing: Tech_Text_Based_Sample.pdf
 Loaded 2 pages

Total documents loaded: 2
Page 1: 1263 characters
Page 2: 1593 characters
'Technology Fundamentals\nA Text-Based PDF for LangChain and RAG Practice\nThis sample document contains selectable, machine-readable text. It is designed for testing PDF loading,\ndocument splitting, embeddings, and retrieval-augmented generation (RAG) workflows.\n1. What is Artificial Intelligence?\nArtificial intelligence (AI) is a field of computer science focused on building systems that perform tasks\ncommonly associated with human intelligence. Examples include understanding language, recognizing\nimages, making predictions, and helping people make decisions.\nMachine learning is a subset of AI. Instead of writing every rule by hand, developers train a model using\ndata. The model learns patterns from examples and uses those patterns to make predictions on new inputs.\n2. What is a Large Language Model?\nA large language 

In [22]:
len(all_pdf_documents)

2

In [23]:
## Text Splitter get into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n"," ",""]
    )


    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} docs into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs    


##    split_documents(all_pdf_documents)    

### embedding and vectorStoreDB


In [24]:
split_documents(all_pdf_documents) 

## 1:02:11

split 2 docs into 4 chunks

Example chunk:
Content: Technology Fundamentals
A Text-Based PDF for LangChain and RAG Practice
This sample document contains selectable, machine-readable text. It is designed for testing PDF loading,
document splitting, emb...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-03T08:20:20+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-03T08:20:20+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\text_files\\Tech_Text_Based_Sample.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Tech_Text_Based_Sample.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-03T08:20:20+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-03T08:20:20+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\text_files\\Tech_Text_Based_Sample.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Tech_Text_Based_Sample.pdf', 'file_type': 'pdf'}, page_content='Technology Fundamentals\nA Text-Based PDF for LangChain and RAG Practice\nThis sample document contains selectable, machine-readable text. It is designed for testing PDF loading,\ndocument splitting, embeddings, and retrieval-augmented generation (RAG) workflows.\n1. What is Artificial Intelligence?\nArtificial intelligence (AI) is a field of computer science focused on building systems that perform tasks\ncommonly associated with human intelligence. Examples include understanding language, recognizing\nim

### embedding and vectorStore db